# Uno et al. 2005 Optik: Digitized Aberrations

This notebook computes the digitized aberration quantities defined by Uno et al., *Optik* 116 (2005), formulas (38)-(47), using simulated probe profiles from the Colab GPU smoke-test workflow.

Reference used for the formulas: local PDF `/Users/yuemingguo/Downloads/Uno et al 2005 Optik.pdf`, page 445.

Important notation: Uno's profile width is written with the Greek letter sigma in formula (45). In this notebook it is named `Xigma` so it is not confused with the Gaussian probe smearing parameter `sigma=2` used during probe image generation.

## 1. Check GPU

In [ ]:
!nvidia-smi

: 

## 2. Clone or Pull Latest GitHub Code

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
ROOT = Path("/content/Aberration-Simulation")

if ROOT.exists():
    print("Repository already exists. Pulling latest main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=ROOT, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
print("repo root:", ROOT)
print("commit:", commit)

## 3. Install and Verify Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if importlib.util.find_spec("cupy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=True)
    raise RuntimeError("Installed CuPy. Restart the Colab runtime, then rerun from the top.")

print("Dependencies are ready.")

## 4. Verify CuPy Backend

In [ ]:
!python scripts/check_backend.py

from aberration_simulation.backend import HAS_CUPY, asnumpy, xp

assert HAS_CUPY, "CuPy is not active. Choose a GPU runtime, restart, and rerun."
print("CuPy GPU smoke path is active.")
print("device count:", xp.cuda.runtime.getDeviceCount())

## 5. Run Probe Simulation

This uses the same smoke-test simulation path as `notebooks/colab_gpu_smoke_test.ipynb`.

In [ ]:
!python scripts/run_smoke_test.py

## 6. Uno Formula Implementation

From the local PDF, formulas (38)-(44):

- `Cdf_value = (1/N) sum_k (Xigma_u,k - Xigma_o,k)`
- `A1_value = (2/N) sum_k (Xigma_u,k - Xigma_o,k) exp(2 i theta_k)`
- `B2_value = (2/N) sum_k (Mu_u,k + Mu_o,k) exp(i theta_k)`
- `A2_value = (2/N) sum_k (Mu_u,k + Mu_o,k) exp(3 i theta_k)`
- `Cs_value = (1/N) sum_k (Rho_u,k - Rho_o,k)`
- `S3_value = (2/N) sum_k (Rho_u,k - Rho_o,k) exp(2 i theta_k)`
- `A3_value = (2/N) sum_k (Xigma_u,k - Xigma_o,k) exp(4 i theta_k)`

Formulas (45)-(47) define line-profile characteristics from profile samples `p_j`, where `j=0` is the probe center, `W=sum_j p_j`, and `T=sum_j p_j^2`:

- `Xigma = sqrt((1/W) sum_j j^2 p_j)`
- `Mu = (1/W) sum_j j p_j`
- `Rho = (Xigma^2/T) sum_{j != 0} ((p_j - p_0) p_j / abs(j))`

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from aberration_simulation.line_profiles import extract_line_profiles_from_stack

OUTPUT_DIR = ROOT / "outputs"
NPZ_PATH = OUTPUT_DIR / "smoke_probe_images.npz"
CSV_PATH = OUTPUT_DIR / "uno_2005_digitized_aberrations.csv"

UNDER_FOCUS_C1_OFFSET = -909
OVER_FOCUS_C1_OFFSET = 909
PROFILE_RADIUS_PIXELS = 80
PROFILE_STEP_DEGREES = 10
NUM_PROFILE_LINES = int(180 / PROFILE_STEP_DEGREES) + 1

print("under-focus C1_offset:", UNDER_FOCUS_C1_OFFSET, "nm")
print("over-focus C1_offset:", OVER_FOCUS_C1_OFFSET, "nm")
print("line-profile angles: 0 to 170 degrees in 10-degree steps")

In [ ]:
def load_smoke_outputs(path):
    data = np.load(path, allow_pickle=True)
    names = [str(name) for name in data["parameter_names"]]
    rows = data["parameters"]
    parameters = [
        {name: float(value) for name, value in zip(names, row)}
        for row in rows
    ]
    return data["probe_images"], parameters


COMBINATION_FIELDS = (
    "C1",
    "A1_amp",
    "A1_phase",
    "A2_amp",
    "A2_phase",
    "B2_amp",
    "B2_phase",
    "A3_amp",
    "A3_phase",
    "C3",
)


def combination_key(params):
    return tuple(params.get(field, 0.0) for field in COMBINATION_FIELDS)


def select_under_over_pairs(parameters):
    pairs = {}
    representatives = {}
    for index, params in enumerate(parameters):
        key = combination_key(params)
        representatives.setdefault(key, params)
        pair = pairs.setdefault(key, {})
        if np.isclose(params["C1_offset"], UNDER_FOCUS_C1_OFFSET):
            pair["under"] = index
        if np.isclose(params["C1_offset"], OVER_FOCUS_C1_OFFSET):
            pair["over"] = index

    selected = []
    for key, pair in pairs.items():
        if "under" in pair and "over" in pair:
            selected.append((representatives[key], pair["under"], pair["over"]))
    return selected


probe_images, parameters = load_smoke_outputs(NPZ_PATH)
pairs = select_under_over_pairs(parameters)

print("probe image stack:", probe_images.shape)
print("under/over pairs:", len(pairs))

In [ ]:
def compute_line_characteristics(profiles_np, radius):
    """Compute Xigma, Mu, and Rho for each angular line profile.

    profiles_np has shape `(num_angles, 2 * radius + 1)`.
    The profile-coordinate index j is measured in pixels, with j=0 at center.
    """
    j = np.arange(-radius, radius + 1, dtype=float)
    center_index = int(np.argmin(np.abs(j)))
    p0 = profiles_np[:, center_index]

    W = np.sum(profiles_np, axis=1)
    T = np.sum(profiles_np ** 2, axis=1)
    W = np.where(W == 0, np.nan, W)
    T = np.where(T == 0, np.nan, T)

    Xigma = np.sqrt(np.sum((j[None, :] ** 2) * profiles_np, axis=1) / W)
    Mu = np.sum(j[None, :] * profiles_np, axis=1) / W

    nonzero = j != 0
    curvature_sum = np.sum(
        ((profiles_np[:, nonzero] - p0[:, None]) * profiles_np[:, nonzero])
        / np.abs(j[nonzero])[None, :],
        axis=1,
    )
    Rho = (Xigma ** 2 / T) * curvature_sum

    return {"Xigma": Xigma, "Mu": Mu, "Rho": Rho}


def compute_uno_values(under_chars, over_chars, angles_deg):
    theta = np.deg2rad(angles_deg)
    N = len(theta)

    Xigma_diff = under_chars["Xigma"] - over_chars["Xigma"]
    Mu_sum = under_chars["Mu"] + over_chars["Mu"]
    Rho_diff = under_chars["Rho"] - over_chars["Rho"]

    Cdf_value = np.sum(Xigma_diff) / N
    A1_value = 2 * np.sum(Xigma_diff * np.exp(2j * theta)) / N
    B2_value = 2 * np.sum(Mu_sum * np.exp(1j * theta)) / N
    A2_value = 2 * np.sum(Mu_sum * np.exp(3j * theta)) / N
    Cs_value = np.sum(Rho_diff) / N
    S3_value = 2 * np.sum(Rho_diff * np.exp(2j * theta)) / N
    A3_value = 2 * np.sum(Xigma_diff * np.exp(4j * theta)) / N

    return {
        "Cdf_value": Cdf_value,
        "A1_value": A1_value,
        "B2_value": B2_value,
        "A2_value": A2_value,
        "Cs_value": Cs_value,
        "S3_value": S3_value,
        "A3_value": A3_value,
    }


UNO_HARMONIC_ORDERS = {
    "A1_value": 2,
    "B2_value": 1,
    "A2_value": 3,
    "S3_value": 2,
    "A3_value": 4,
}

PRIMARY_PHASE_CONVENTION = "y_down_sign_flipped"


def wrap_period_deg(angle_deg, period_deg):
    return float(np.mod(angle_deg, period_deg))


def harmonic_orientation_deg(phase_deg, order):
    return wrap_period_deg(phase_deg / order, 360.0 / order)


def interpreted_harmonic_phase_deg(raw_complex_phase_deg, order):
    """Return the physical orientation using the adopted Uno convention.

    Formulae (39)-(44) produce complex harmonic phases. To compare with the
    input aberration angle in displayed probe images, we convert from harmonic
    phase to orientation, flip the image y-axis, and apply the under/over sign
    convention selected by the A1 phase diagnostics.
    """
    return harmonic_orientation_deg(-(raw_complex_phase_deg + 180.0), order)


def add_complex_columns(row, name, value):
    raw_phase_deg = float(np.angle(value, deg=True))
    row[name] = str(value)
    row[name + "_real"] = float(np.real(value))
    row[name + "_imag"] = float(np.imag(value))
    row[name + "_abs"] = float(np.abs(value))
    row[name + "_complex_phase_deg"] = raw_phase_deg
    row[name + "_raw_phase_deg"] = raw_phase_deg

    # For m-fold angular terms, the complex phase is m times the physical
    # orientation angle. The primary `*_phase_deg` column uses the convention
    # that matches the input A1 phase: displayed y-down coordinates plus the
    # under/over sign flip. Raw complex phase remains available above.
    order = UNO_HARMONIC_ORDERS.get(name)
    if order is not None:
        period_deg = 360.0 / order
        orientation_deg = harmonic_orientation_deg(raw_phase_deg, order)
        orientation_y_down_deg = harmonic_orientation_deg(-raw_phase_deg, order)
        orientation_sign_flipped_deg = harmonic_orientation_deg(raw_phase_deg + 180.0, order)
        orientation_y_down_sign_flipped_deg = interpreted_harmonic_phase_deg(raw_phase_deg, order)

        row[name + "_orientation_deg"] = orientation_deg
        row[name + "_orientation_y_down_deg"] = orientation_y_down_deg
        row[name + "_orientation_sign_flipped_deg"] = orientation_sign_flipped_deg
        row[name + "_orientation_y_down_sign_flipped_deg"] = orientation_y_down_sign_flipped_deg
        row[name + "_orientation_period_deg"] = period_deg
        row[name + "_phase_convention"] = PRIMARY_PHASE_CONVENTION
        row[name + "_phase_period_deg"] = period_deg
        row[name + "_phase_deg"] = orientation_y_down_sign_flipped_deg
        row[name + "_interpreted_phase_deg"] = orientation_y_down_sign_flipped_deg
        row[name + "_interpreted_phase_period_deg"] = period_deg
    else:
        row[name + "_phase_deg"] = raw_phase_deg


## 7. Compute Uno Digitized Aberrations

In [ ]:
rows = []
characteristics_by_case = []

for case_index, (params, under_index, over_index) in enumerate(pairs):
    stack = probe_images[:, :, [under_index, over_index]]
    profiles, coords = extract_line_profiles_from_stack(
        stack,
        num_lines=NUM_PROFILE_LINES,
        radius=PROFILE_RADIUS_PIXELS,
    )
    profiles_np = asnumpy(profiles)
    angles_deg = np.asarray(coords["angles_deg"], dtype=float)

    under_chars = compute_line_characteristics(profiles_np[:, :, 0], PROFILE_RADIUS_PIXELS)
    over_chars = compute_line_characteristics(profiles_np[:, :, 1], PROFILE_RADIUS_PIXELS)
    uno_values = compute_uno_values(under_chars, over_chars, angles_deg)

    row = {field: params.get(field, 0.0) for field in COMBINATION_FIELDS}
    row["case_index"] = case_index
    row["under_index"] = under_index
    row["over_index"] = over_index
    row["under_C1_offset"] = UNDER_FOCUS_C1_OFFSET
    row["over_C1_offset"] = OVER_FOCUS_C1_OFFSET
    for name, value in uno_values.items():
        add_complex_columns(row, name, value)
    rows.append(row)

    characteristics_by_case.append({
        "params": params,
        "angles_deg": angles_deg,
        "under_chars": under_chars,
        "over_chars": over_chars,
        "uno_values": uno_values,
    })

print("computed cases:", len(rows))
print("first case Uno values:")
for key, value in characteristics_by_case[0]["uno_values"].items():
    print("  ", key, value)

## 8. Save Results

In [ ]:
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
fieldnames = list(rows[0].keys())
with CSV_PATH.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("saved:", CSV_PATH)
print("columns:", len(fieldnames))

## 9. Interpret and Visualize A1 Harmonic Phases

The raw complex phase is stored as `A1_value_complex_phase_deg` / `A1_value_raw_phase_deg`. The primary reported column `A1_value_phase_deg` now applies the selected physical convention: convert the doubled harmonic phase to orientation, use displayed y-down image coordinates, and apply the under/over sign flip. This should be directly comparable with `A1_phase` modulo 180 degrees.


In [ ]:
def circular_difference_deg(a, b, period):
    return np.abs((a - b + period / 2) % period - period / 2)


A1_ORIENTATION_CANDIDATES = [
    "orientation_deg",
    "orientation_y_down_deg",
    "orientation_sign_flipped_deg",
    "orientation_y_down_sign_flipped_deg",
]

PRIMARY_PHASE_COLUMNS = [
    "A1_value_phase_deg",
    "B2_value_phase_deg",
    "A2_value_phase_deg",
    "S3_value_phase_deg",
    "A3_value_phase_deg",
]

RAW_PHASE_COLUMNS = [
    "A1_value_complex_phase_deg",
    "B2_value_complex_phase_deg",
    "A2_value_complex_phase_deg",
    "S3_value_complex_phase_deg",
    "A3_value_complex_phase_deg",
]


def a1_phase_diagnostic(rows):
    diagnostics = []
    for row in rows:
        if not np.isclose(row.get("A1_amp", 0.0), 0.0):
            target = row["A1_phase"] % 180.0
            candidates = {
                "orientation_deg": row["A1_value_orientation_deg"],
                "orientation_y_down_deg": row["A1_value_orientation_y_down_deg"],
                "orientation_sign_flipped_deg": row["A1_value_orientation_sign_flipped_deg"],
                "orientation_y_down_sign_flipped_deg": row["A1_value_orientation_y_down_sign_flipped_deg"],
                "primary_phase_deg": row["A1_value_phase_deg"],
            }
            errors = {
                name + "_error": circular_difference_deg(value, target, 180.0)
                for name, value in candidates.items()
            }
            diagnostics.append({
                "case_index": row["case_index"],
                "A1_amp": row["A1_amp"],
                "A1_phase_input": row["A1_phase"],
                "A1_phase_input_mod180": target,
                "A1_value_complex_phase_deg_raw": row["A1_value_complex_phase_deg"],
                **candidates,
                **errors,
            })
    return diagnostics


def summarize_a1_phase_diagnostic(diagnostics):
    summary = []
    for name in A1_ORIENTATION_CANDIDATES + ["primary_phase_deg"]:
        errors = np.asarray([item[name + "_error"] for item in diagnostics], dtype=float)
        summary.append({
            "candidate": name,
            "mean_abs_error_deg": float(np.nanmean(errors)),
            "median_abs_error_deg": float(np.nanmedian(errors)),
            "max_abs_error_deg": float(np.nanmax(errors)),
        })
    return sorted(summary, key=lambda item: item["median_abs_error_deg"])


a1_diagnostics = a1_phase_diagnostic(rows)
a1_summary = summarize_a1_phase_diagnostic(a1_diagnostics)

print("Primary phase convention:", PRIMARY_PHASE_CONVENTION)
print("Primary phase columns written to CSV:")
for column in PRIMARY_PHASE_COLUMNS:
    print("  ", column)
print("Raw complex phase columns retained in CSV:")
for column in RAW_PHASE_COLUMNS:
    print("  ", column)

print()
print("A1 orientation convention summary across", len(a1_diagnostics), "A1 cases:")
for item in a1_summary:
    print(item)

labels = [item["candidate"].replace("orientation_", "").replace("_deg", "") for item in a1_summary]
medians = [item["median_abs_error_deg"] for item in a1_summary]
means = [item["mean_abs_error_deg"] for item in a1_summary]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(labels))
ax.bar(x - 0.18, medians, width=0.36, label="median error")
ax.bar(x + 0.18, means, width=0.36, label="mean error")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_ylabel("absolute error vs input A1_phase (deg)")
ax.set_title("A1 orientation convention check")
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
axes = axes.ravel()
target = np.asarray([item["A1_phase_input_mod180"] for item in a1_diagnostics])
amp = np.asarray([item["A1_amp"] for item in a1_diagnostics])
for axis, name in zip(axes, A1_ORIENTATION_CANDIDATES):
    values = np.asarray([item[name] for item in a1_diagnostics])
    sc = axis.scatter(target, values, c=amp, cmap="viridis", s=24, alpha=0.85)
    axis.plot([0, 180], [0, 180], color="black", linewidth=1, linestyle="--")
    axis.set_title(name)
    axis.set_xlabel("input A1_phase mod 180 (deg)")
    axis.set_ylabel("derived orientation (deg)")
    axis.set_xlim(-5, 185)
    axis.set_ylim(-5, 185)
    axis.grid(alpha=0.3)
fig.colorbar(sc, ax=axes.tolist(), label="A1_amp")
fig.suptitle("Derived A1 orientation candidates vs input phase")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
primary = np.asarray([item["primary_phase_deg"] for item in a1_diagnostics])
sc = ax.scatter(target, primary, c=amp, cmap="viridis", s=28, alpha=0.9)
ax.plot([0, 180], [0, 180], color="black", linewidth=1, linestyle="--")
ax.set_title("Primary reported A1 phase")
ax.set_xlabel("input A1_phase mod 180 (deg)")
ax.set_ylabel("A1_value_phase_deg")
ax.set_xlim(-5, 185)
ax.set_ylim(-5, 185)
ax.grid(alpha=0.3)
fig.colorbar(sc, ax=ax, label="A1_amp")
fig.tight_layout()
plt.show()


## 9. Quick Visual Check

In [ ]:
def find_first_case(**criteria):
    for index, item in enumerate(characteristics_by_case):
        params = item["params"]
        if all(np.isclose(params.get(key, 0.0), value) for key, value in criteria.items()):
            return index, item
    return 0, characteristics_by_case[0]


case_index, item = find_first_case(A1_amp=60, A1_phase=157.5)
angles_deg = item["angles_deg"]
row = rows[case_index]

fig, axes = plt.subplots(3, 1, figsize=(8, 9), sharex=True)
for axis, metric in zip(axes, ["Xigma", "Mu", "Rho"]):
    axis.plot(angles_deg, item["under_chars"][metric], marker="o", label="under")
    axis.plot(angles_deg, item["over_chars"][metric], marker="s", label="over")
    axis.set_ylabel(metric)
    axis.grid(True, alpha=0.3)
    axis.legend()
axes[-1].set_xlabel("line angle (deg)")
fig.suptitle("Uno line characteristics, case {}".format(case_index))
fig.tight_layout()
plt.show()

print("case parameters:", item["params"])
print("Uno values:")
for key, value in item["uno_values"].items():
    primary_phase = row.get(key + "_phase_deg")
    raw_phase = row.get(key + "_complex_phase_deg")
    period = row.get(key + "_phase_period_deg")
    print("  {} = {} | abs={} primary_phase_deg={} raw_complex_phase_deg={} period={}".format(
        key,
        value,
        np.abs(value),
        primary_phase,
        raw_phase,
        period,
    ))
